In [17]:

#1. Install the medmnist Library
%pip install medmnist
#2. Python Commands to Load the Dataset
import medmnist
from medmnist import PneumoniaMNIST
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Define any transformations (e.g., convert to PyTorch tensors)
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# 1. Download and load the datasets
train_dataset = PneumoniaMNIST(split='train', transform=data_transform, download=True)
val_dataset = PneumoniaMNIST(split='val', transform=data_transform, download=True)
test_dataset = PneumoniaMNIST(split='test', transform=data_transform, download=True)

# 2. Wrap them in PyTorch DataLoaders for your model
batch_size = 128
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)


In [18]:
import torch
import torch.nn as nn #import neural network for torch
import torch.optim as optim #import optimizer for neural network

# build CNN from nn.Module
class xrayCNN(nn.Module):
    def __init__(self):
        super().__init__() # initialize

        # first feature scan with max pooling
        # Conv2d creates sliding window of 3x3 pixels that scan the image for basic patterns like edges. It uses 1 channel as images are greyscale and outputs 16 feature maps.
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        # ReLU function (Rectified Linear Unit) turns negative numbers to 0.
        self.relu1 = nn.ReLU()
        # maxPool shrinks the image size from 28x28 down to 14x14
        self.pool1 = nn.MaxPool2d(2, 2)

        # run a second feature scan with max pooling.
        # takes the 16 outputs from the last layer and expands to 32.
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        # shrinks the image from 14x14 down to 7x7
        self.pool2 = nn.MaxPool2d(2, 2)

        # third feature scan and max pooling
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu4 = nn.ReLU()
        # shrinks the image from 7x7 down to 3x3
        self.pool3 = nn.MaxPool2d(2, 2)

        # linear layers need a flat list of numbers, not a 2D image grid.
        self.fc1 = nn.Linear(576, 64)
        self.relu3 = nn.ReLU()

        # final output layer. funnels the 64 down to just 1 number.
        # positive is for pneumonia and negative is for normal label
        self.fc2 = nn.Linear(64, 1)

    # The forward function defines the path the image takes through the layers
    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu4(self.conv3(x))) # pass through the third block

        x = x.view(-1, 576) # Flatten to 576

        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

# use Colab's GPU
device = torch.device('cuda')
# Create model and immediately pass to GPU
model = xrayCNN().to(device)

# The loss function calculates the error for each round
loss_fn = nn.BCEWithLogitsLoss()

# optimizer adjusts the weights based on error rate
# added weight_decay to penalize large weights and reduce overfitting
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# function to evaulate model performance
def evaluate(model, dataloader):
    # .eval() locks the model's weights
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for images, labels in dataloader:
            # move data to the GPU
            images, labels = images.to(device), labels.to(device).float()

            # get model predictions
            preds = model(images)
            # calculate the error
            loss = loss_fn(preds, labels)
            total_loss += loss.item() * images.size(0)

            # when raw score is >= 0, it predicted to be pneumonia
            binary_preds = (preds >= 0.0).float()
            # determine accuract for predictions
            correct += (binary_preds == labels).sum().item()
            total += labels.size(0)

    # average error and accuracy percentage
    return total_loss / total, correct / total

In [19]:
#train model with 5 epochs using only training data
num_epochs = 5

for epoch in range(num_epochs):
    # set model to model.train() so it can learn
    model.train()
    running_loss = 0.0

    # loop through the training data one batch at a time, passing both labels and images
    for images, labels in train_loader:

        images, labels = images.to(device), labels.to(device).float()

        # remove calculated gradients from the previous batch
        optimizer.zero_grad()

        # forward pass to predicate see what the model guesses
        preds = model(images)

        # determine error
        loss = loss_fn(preds, labels)

        # backward pass to identify which weights to adjust
        loss.backward()

        # optimize weights
        optimizer.step()

        # update running tally of the error
        running_loss += loss.item() * images.size(0)

    # average error for this epoch
    train_loss = running_loss / len(train_dataset)

    # evaluate function on the validation data
    val_loss, val_acc = evaluate(model, val_loader)

    # print results for this epoch
    print(f"epoch [{epoch+1}/{num_epochs}] train_loss: {train_loss}, val_loss: {val_loss}, val_acc: {val_acc*100}%")



epoch [1/5] train_loss: 0.5771218565630528, val_loss: 0.4957564022704845, val_acc: 74.23664122137404%
epoch [2/5] train_loss: 0.34661006966737135, val_loss: 0.24133171340208928, val_acc: 91.22137404580153%
epoch [3/5] train_loss: 0.19321610637407943, val_loss: 0.16525898424496177, val_acc: 94.27480916030534%
epoch [4/5] train_loss: 0.1633871542440194, val_loss: 0.1552073081941095, val_acc: 94.46564885496184%
epoch [5/5] train_loss: 0.1471263429018635, val_loss: 0.14511206239916896, val_acc: 94.65648854961832%


In [20]:
# Use model on held out test data to determine actual performance
test_loss, test_acc = evaluate(model, test_loader)
print(f"Test data score: {test_acc*100}%")

Test data score: 88.46153846153845%
